# OM_BKK_V1 — conditional rainfall intensity models

This notebook adds a second stage to the existing four rain/no-rain models without
modifying or retraining them.

For each 1, 2, 3, and 6 hour window it:

1. Calculates the maximum hourly precipitation inside the future window, separately
   per station/grid and in observation-time order.
2. Keeps only rows whose future maximum is greater than zero for intensity training.
3. Predicts **Light rain**, **Moderate rain**, or **Heavy rain** conditionally on rain.
4. Uses chronological train/validation/test splits with a horizon-aware purge gap so
   future target periods cannot overlap across splits.
5. Saves models, mappings, features, thresholds, validation/test results, and metadata.

The original notebook and its saved binary models are read-only inputs to the final
two-stage inference example.

## 1. Imports and configuration

Edit `RAINFALL_THRESHOLDS_MM` to change the class boundaries. With the defaults,
a future-window maximum of 8.0 mm is Heavy rain, matching the requested example.

In [ ]:
import json
import os
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

pd.set_option("display.max_columns", 180)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
DB_CONFIG = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE", "postgres"),
    "user": os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", "Pass1234"),
}

SOURCE_TABLE_NAME = '"OM_BKK_DATA"'
PRECOMPUTE_TABLE_NAME = '"OM_BKK_DATA_PRECOMPUTE"'
PROJECT_ROOT = Path.cwd()
BINARY_MODEL_DIR = PROJECT_ROOT / "ML_Model_V2" / "trained_models" / "om_bkk_rain_any_final_4_neighbor_models"
MODEL_DIR = PROJECT_ROOT / "ML_Model_V2" / "trained_models" / "OM_BKK_V1"

HORIZONS = [1, 2, 3, 6]
RAINFALL_THRESHOLDS_MM = {
    "rain_positive_min_exclusive_mm": 0.0,
    "moderate_min_mm": 2.5,
    "heavy_min_mm": 7.5,
}
INTENSITY_LABELS = ["Light rain", "Moderate rain", "Heavy rain"]
LABEL_TO_ID = {label: class_id for class_id, label in enumerate(INTENSITY_LABELS)}
ID_TO_LABEL = {class_id: label for label, class_id in LABEL_TO_ID.items()}

TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
SAMPLE_ROWS = None  # For a quick smoke test only; keep None for final training.
RANDOM_STATE = 42

assert 0.0 <= RAINFALL_THRESHOLDS_MM["rain_positive_min_exclusive_mm"]
assert RAINFALL_THRESHOLDS_MM["moderate_min_mm"] < RAINFALL_THRESHOLDS_MM["heavy_min_mm"]

## 2. Reuse the existing feature set

These columns are copied from `bkk_final_4_rain_any_neighbor_models.ipynb`. No feature,
threshold, or fitted object belonging to the current binary pipeline is changed.

In [ ]:
BASELINE_FEATURE_COLUMNS = [
    "temperature_2m", "relative_humidity_2m", "pressure_msl", "surface_pressure",
    "dew_point_2m", "precipitation", "cloud_cover", "wind_speed_10m",
    "wind_direction_10m", "temperature_dew_point_spread", "pressure_msl_change_3h",
    "pressure_msl_change_6h", "precipitation_lag_1h", "precipitation_lag_2h",
    "precipitation_lag_3h", "precipitation_lag_6h", "precipitation_sum_past_3h",
    "precipitation_sum_past_6h", "precipitation_sum_past_12h", "precipitation_sum_past_24h",
    "cloud_cover_lag_1h", "cloud_cover_lag_3h", "cloud_cover_lag_6h",
    "humidity_lag_1h", "humidity_lag_3h", "humidity_lag_6h",
    "wind_speed_lag_1h", "wind_speed_lag_3h", "hour_sin", "hour_cos",
    "month_sin", "month_cos", "grid_row", "grid_column", "latitude", "longitude",
]

NEIGHBOR_FEATURE_COLUMNS = [
    "neighbor_count", "neighbor_precipitation_mean", "neighbor_precipitation_max",
    "neighbor_precipitation_sum", "neighbor_rain_count", "neighbor_rain_rate",
    "neighbor_cloud_cover_mean", "neighbor_cloud_cover_max", "neighbor_relative_humidity_mean",
    "neighbor_relative_humidity_max", "neighbor_pressure_msl_mean", "neighbor_pressure_msl_min",
    "neighbor_pressure_msl_max", "neighbor_temperature_2m_mean", "neighbor_dew_point_2m_mean",
    "neighbor_temperature_dew_point_spread_mean", "neighbor_wind_speed_10m_mean",
    "neighbor_wind_speed_10m_max", "row_minus_precipitation_mean", "row_plus_precipitation_mean",
    "column_minus_precipitation_mean", "column_plus_precipitation_mean", "row_minus_cloud_cover_mean",
    "row_plus_cloud_cover_mean", "column_minus_cloud_cover_mean", "column_plus_cloud_cover_mean",
    "neighbor_precipitation_mean_minus_center", "neighbor_cloud_cover_mean_minus_center",
    "neighbor_relative_humidity_mean_minus_center", "center_pressure_msl_minus_neighbor_mean",
]

FEATURE_COLUMNS = BASELINE_FEATURE_COLUMNS + NEIGHBOR_FEATURE_COLUMNS
TARGET_COLUMNS = [f"intensity_{horizon}h" for horizon in HORIZONS]
FUTURE_MAX_COLUMNS = [f"future_max_precipitation_{horizon}h_mm" for horizon in HORIZONS]

print(f"Features reused: {len(FEATURE_COLUMNS)}")
print(f"New targets: {TARGET_COLUMNS}")

## 3. Inspect the database and read training rows

The query is ordered by station/grid and observation time. Target construction below
uses `groupby(grid_number).shift(...)` and verifies that every shifted row is exactly
the requested number of hours ahead; gaps are therefore not mistaken for future hours.

In [ ]:
def connect():
    return psycopg2.connect(**DB_CONFIG)


def inspect_database_schema():
    query = '''
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'public' AND table_name = 'OM_BKK_DATA_PRECOMPUTE'
    ORDER BY ordinal_position
    '''
    with connect() as conn:
        schema = pd.read_sql_query(query, conn)
        summary = pd.read_sql_query(
            f'''SELECT COUNT(*) AS rows,
                       COUNT(DISTINCT grid_number) AS stations,
                       MIN(local_forecast_time) AS min_time,
                       MAX(local_forecast_time) AS max_time
                FROM {PRECOMPUTE_TABLE_NAME}''',
            conn,
        )
    return schema, summary


schema_df, database_summary = inspect_database_schema()
display(database_summary)
display(schema_df)

required_database_columns = {"grid_number", "local_forecast_time", *FEATURE_COLUMNS}
missing_database_columns = sorted(required_database_columns - set(schema_df["column_name"]))
if missing_database_columns:
    raise KeyError(f"Missing required database columns: {missing_database_columns}")

In [ ]:
def read_training_data(sample_rows=None):
    select_columns = [
        "grid_number",
        "local_forecast_time AS forecast_time",
        *FEATURE_COLUMNS,
    ]
    select_sql = ",\n        ".join(select_columns)
    query = f'''
    SELECT
        {select_sql}
    FROM {PRECOMPUTE_TABLE_NAME}
    WHERE pressure_msl_change_6h IS NOT NULL
      AND precipitation_lag_6h IS NOT NULL
      AND precipitation_sum_past_24h IS NOT NULL
      AND cloud_cover_lag_6h IS NOT NULL
      AND humidity_lag_6h IS NOT NULL
      AND wind_speed_lag_3h IS NOT NULL
      AND neighbor_count > 0
    ORDER BY grid_number, local_forecast_time
    '''
    if sample_rows:
        query = f'''
        SELECT *
        FROM ({query}) complete_history_rows
        ORDER BY forecast_time, grid_number
        LIMIT {int(sample_rows)}
        '''
    with connect() as conn:
        data = pd.read_sql_query(query, conn, parse_dates=["forecast_time"])
    data = data.sort_values(["grid_number", "forecast_time"], kind="stable").reset_index(drop=True)
    data[FEATURE_COLUMNS] = data[FEATURE_COLUMNS].astype("float32")
    return data


df = read_training_data(SAMPLE_ROWS)
print(df.shape)
print(df["forecast_time"].min(), "to", df["forecast_time"].max())
display(df.head())

## 4. Create maximum-within-window intensity targets

`intensity_1h`, `intensity_2h`, `intensity_3h`, and `intensity_6h` are categorical
targets. A zero future maximum is stored as a missing intensity target and is never
passed to an intensity model.

In [ ]:
def intensity_labels_from_max(future_max_mm):
    moderate_min = RAINFALL_THRESHOLDS_MM["moderate_min_mm"]
    heavy_min = RAINFALL_THRESHOLDS_MM["heavy_min_mm"]
    values = np.select(
        [
            (future_max_mm > 0) & (future_max_mm < moderate_min),
            (future_max_mm >= moderate_min) & (future_max_mm < heavy_min),
            future_max_mm >= heavy_min,
        ],
        INTENSITY_LABELS,
        default=None,
    )
    return pd.Categorical(values, categories=INTENSITY_LABELS, ordered=True)


def add_future_intensity_targets(data):
    out = data.sort_values(["grid_number", "forecast_time"], kind="stable").reset_index(drop=True)
    station_groups = out.groupby("grid_number", sort=False)
    future_precipitation = {}

    for step in range(1, max(HORIZONS) + 1):
        shifted_precipitation = station_groups["precipitation"].shift(-step)
        shifted_time = station_groups["forecast_time"].shift(-step)
        expected_time = out["forecast_time"] + pd.Timedelta(hours=step)
        future_precipitation[step] = shifted_precipitation.where(shifted_time.eq(expected_time)).astype("float32")

    for horizon in HORIZONS:
        future_frame = pd.concat(
            [future_precipitation[step] for step in range(1, horizon + 1)],
            axis=1,
        )
        future_max_column = f"future_max_precipitation_{horizon}h_mm"
        target_column = f"intensity_{horizon}h"
        out[future_max_column] = future_frame.max(axis=1, skipna=False).astype("float32")
        out[target_column] = intensity_labels_from_max(out[future_max_column])

    return out


df = add_future_intensity_targets(df)
# Require a complete six-hour future window so all four targets share the same base rows.
df = df[df["future_max_precipitation_6h_mm"].notna()].reset_index(drop=True)

example = pd.Series([0, 0.5, 1.2, 8.0, 2.0, 0], dtype="float32")
example_max = float(example.max())
example_class = str(intensity_labels_from_max(pd.Series([example_max]))[0])
print(f"Example maximum: {example_max:.1f} mm -> {example_class}")
display(df[["grid_number", "forecast_time", *FUTURE_MAX_COLUMNS, *TARGET_COLUMNS]].head(10))

In [ ]:
balance_rows = []
for horizon in HORIZONS:
    target = f"intensity_{horizon}h"
    future_max = f"future_max_precipitation_{horizon}h_mm"
    rainy = df[future_max] > RAINFALL_THRESHOLDS_MM["rain_positive_min_exclusive_mm"]
    counts = df.loc[rainy, target].value_counts().reindex(INTENSITY_LABELS, fill_value=0)
    for class_label, rows in counts.items():
        balance_rows.append({
            "horizon_h": horizon,
            "target_column": target,
            "class_id": LABEL_TO_ID[class_label],
            "class_label": class_label,
            "rows": int(rows),
            "conditional_class_rate": float(rows / max(rainy.sum(), 1)),
            "rainy_training_pool_rows": int(rainy.sum()),
            "all_complete_rows": int(len(df)),
        })

target_balance = pd.DataFrame(balance_rows)
display(target_balance)

sns.barplot(data=target_balance, x="horizon_h", y="conditional_class_rate", hue="class_label")
plt.title("Conditional Future-Window Intensity Balance")
plt.xlabel("Future window (hours)")
plt.ylabel("Rate among future-rain rows")
plt.show()

## 5. Leakage-safe chronological splits

All stations at a given timestamp stay in the same split. For an `h`-hour model,
the final `h-1` observation times before each boundary are purged from the earlier
split. This makes the latest target hour in the earlier split strictly earlier than
the first target hour in the later split.

In [ ]:
unique_times = np.array(sorted(df["forecast_time"].unique()))
train_boundary = pd.Timestamp(unique_times[int(len(unique_times) * TRAIN_FRACTION)])
validation_boundary = pd.Timestamp(unique_times[int(len(unique_times) * (TRAIN_FRACTION + VALIDATION_FRACTION))])


def purged_time_split_masks(data, horizon):
    purge = pd.Timedelta(hours=max(horizon - 1, 0))
    time_values = data["forecast_time"]
    masks = {
        "train": time_values < (train_boundary - purge),
        "validation": (time_values >= train_boundary) & (time_values < (validation_boundary - purge)),
        "test": time_values >= validation_boundary,
    }

    nonempty = {name: data.loc[mask, "forecast_time"] for name, mask in masks.items()}
    if any(values.empty for values in nonempty.values()):
        raise ValueError(f"Horizon {horizon}h has an empty split")

    latest_train_target_time = nonempty["train"].max() + pd.Timedelta(hours=horizon)
    first_validation_target_time = nonempty["validation"].min() + pd.Timedelta(hours=1)
    latest_validation_target_time = nonempty["validation"].max() + pd.Timedelta(hours=horizon)
    first_test_target_time = nonempty["test"].min() + pd.Timedelta(hours=1)
    assert latest_train_target_time < first_validation_target_time
    assert latest_validation_target_time < first_test_target_time
    return masks


split_masks_by_horizon = {horizon: purged_time_split_masks(df, horizon) for horizon in HORIZONS}
split_rows = []
for horizon, masks in split_masks_by_horizon.items():
    rainy = df[f"future_max_precipitation_{horizon}h_mm"] > 0
    for split_name, mask in masks.items():
        split_rows.append({
            "horizon_h": horizon,
            "split": split_name,
            "all_rows_after_purge": int(mask.sum()),
            "rainy_intensity_rows": int((mask & rainy).sum()),
            "min_observation_time": str(df.loc[mask, "forecast_time"].min()),
            "max_observation_time": str(df.loc[mask, "forecast_time"].max()),
        })

split_summary = pd.DataFrame(split_rows)
print(f"Train boundary: {train_boundary}")
print(f"Validation boundary: {validation_boundary}")
display(split_summary)

## 6. Train one conditional intensity model per horizon

The `rainy` mask is applied before extracting every train, validation, and test set.
Consequently, no zero-rain row can enter intensity fitting or intensity evaluation.

In [ ]:
def encode_target(series):
    encoded = series.map(LABEL_TO_ID)
    if encoded.isna().any():
        raise ValueError("Intensity target contains an unlabelled row")
    return encoded.astype("int8")


intensity_models = {}
training_rows = []

for horizon in HORIZONS:
    target_column = f"intensity_{horizon}h"
    future_max_column = f"future_max_precipitation_{horizon}h_mm"
    masks = split_masks_by_horizon[horizon]
    rainy = df[future_max_column] > RAINFALL_THRESHOLDS_MM["rain_positive_min_exclusive_mm"]

    train_mask = masks["train"] & rainy
    validation_mask = masks["validation"] & rainy
    x_train = df.loc[train_mask, FEATURE_COLUMNS]
    y_train = encode_target(df.loc[train_mask, target_column])
    x_validation = df.loc[validation_mask, FEATURE_COLUMNS]
    y_validation = encode_target(df.loc[validation_mask, target_column])

    missing_training_classes = sorted(set(range(len(INTENSITY_LABELS))) - set(y_train.unique()))
    if missing_training_classes:
        raise ValueError(f"Horizon {horizon}h is missing training classes: {missing_training_classes}")

    print(f"Training intensity_model_{horizon}h on {len(y_train):,} rain-positive rows...")
    model = LGBMClassifier(
        objective="multiclass",
        num_class=len(INTENSITY_LABELS),
        class_weight="balanced",
        n_estimators=400,
        learning_rate=0.04,
        num_leaves=63,
        min_child_samples=80,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(
        x_train,
        y_train,
        eval_set=[(x_validation, y_validation)],
        eval_metric="multi_logloss",
        callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
    )
    intensity_models[horizon] = model
    training_rows.append({
        "horizon_h": horizon,
        "train_rows": int(len(y_train)),
        "validation_rows": int(len(y_validation)),
        "best_iteration": int(model.best_iteration_ or model.n_estimators),
    })

intensity_model_1h = intensity_models[1]
intensity_model_2h = intensity_models[2]
intensity_model_3h = intensity_models[3]
intensity_model_6h = intensity_models[6]
display(pd.DataFrame(training_rows))

## 7. Evaluate conditional intensity

PR-AUC is calculated independently for Light, Moderate, and Heavy using one-vs-rest.
The aggregate table also exposes Heavy-rain precision and recall directly.

In [ ]:
def aligned_predict_proba(model, x):
    raw = model.predict_proba(x)
    aligned = np.zeros((len(x), len(INTENSITY_LABELS)), dtype="float64")
    for source_column, class_id in enumerate(model.classes_):
        aligned[:, int(class_id)] = raw[:, source_column]
    return aligned


def evaluate_intensity_model(model, x, y_true, horizon, split):
    probabilities = aligned_predict_proba(model, x)
    y_true_array = np.asarray(y_true, dtype="int8")
    y_pred = probabilities.argmax(axis=1).astype("int8")

    per_class_rows = []
    for class_id, class_label in enumerate(INTENSITY_LABELS):
        binary_true = (y_true_array == class_id).astype("int8")
        binary_pred = (y_pred == class_id).astype("int8")
        pr_auc = (
            float(average_precision_score(binary_true, probabilities[:, class_id]))
            if binary_true.sum() > 0
            else np.nan
        )
        per_class_rows.append({
            "horizon_h": horizon,
            "split": split,
            "class_id": class_id,
            "class_label": class_label,
            "support": int(binary_true.sum()),
            "precision": float(precision_score(binary_true, binary_pred, zero_division=0)),
            "recall": float(recall_score(binary_true, binary_pred, zero_division=0)),
            "f1": float(f1_score(binary_true, binary_pred, zero_division=0)),
            "pr_auc_ovr": pr_auc,
        })

    per_class = pd.DataFrame(per_class_rows)
    heavy = per_class[per_class["class_label"] == "Heavy rain"].iloc[0]
    aggregate = {
        "horizon_h": horizon,
        "split": split,
        "rows": int(len(y_true_array)),
        "macro_f1": float(f1_score(y_true_array, y_pred, labels=range(len(INTENSITY_LABELS)), average="macro", zero_division=0)),
        "macro_precision": float(precision_score(y_true_array, y_pred, labels=range(len(INTENSITY_LABELS)), average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true_array, y_pred, labels=range(len(INTENSITY_LABELS)), average="macro", zero_division=0)),
        "heavy_rain_precision": float(heavy["precision"]),
        "heavy_rain_recall": float(heavy["recall"]),
    }

    matrix = confusion_matrix(y_true_array, y_pred, labels=range(len(INTENSITY_LABELS)))
    confusion_rows = []
    for actual_id, actual_label in enumerate(INTENSITY_LABELS):
        for predicted_id, predicted_label in enumerate(INTENSITY_LABELS):
            confusion_rows.append({
                "horizon_h": horizon,
                "split": split,
                "actual_class_id": actual_id,
                "actual_class_label": actual_label,
                "predicted_class_id": predicted_id,
                "predicted_class_label": predicted_label,
                "count": int(matrix[actual_id, predicted_id]),
            })
    return aggregate, per_class_rows, confusion_rows


aggregate_rows = []
class_rows = []
confusion_rows = []

for horizon, model in intensity_models.items():
    rainy = df[f"future_max_precipitation_{horizon}h_mm"] > 0
    for split_name in ["validation", "test"]:
        evaluation_mask = split_masks_by_horizon[horizon][split_name] & rainy
        x_evaluation = df.loc[evaluation_mask, FEATURE_COLUMNS]
        y_evaluation = encode_target(df.loc[evaluation_mask, f"intensity_{horizon}h"])
        aggregate, by_class, confusion = evaluate_intensity_model(
            model, x_evaluation, y_evaluation, horizon, split_name
        )
        aggregate_rows.append(aggregate)
        class_rows.extend(by_class)
        confusion_rows.extend(confusion)

intensity_results = pd.DataFrame(aggregate_rows)
intensity_class_results = pd.DataFrame(class_rows)
intensity_confusion_matrices = pd.DataFrame(confusion_rows)

display(intensity_results.sort_values(["split", "horizon_h"]))
display(intensity_class_results.sort_values(["split", "horizon_h", "class_id"]))
display(intensity_confusion_matrices[intensity_confusion_matrices["split"] == "test"])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)
test_overall = intensity_results[intensity_results["split"] == "test"]
sns.lineplot(data=test_overall, x="horizon_h", y="macro_f1", marker="o", ax=axes[0])
axes[0].set_title("Test Macro F1")
sns.lineplot(data=test_overall, x="horizon_h", y="heavy_rain_precision", marker="o", ax=axes[1])
axes[1].set_title("Test Heavy-rain Precision")
sns.lineplot(data=test_overall, x="horizon_h", y="heavy_rain_recall", marker="o", ax=axes[2])
axes[2].set_title("Test Heavy-rain Recall")
for axis in axes:
    axis.set_xlabel("Future window (hours)")
plt.show()

sns.catplot(
    data=intensity_class_results[intensity_class_results["split"] == "test"],
    x="horizon_h",
    y="pr_auc_ovr",
    hue="class_label",
    kind="bar",
    height=5,
    aspect=1.8,
)
plt.title("Test One-vs-Rest PR-AUC by Intensity Class")
plt.xlabel("Future window (hours)")
plt.ylabel("PR-AUC")
plt.show()

## 8. Save every required artifact

The files are isolated under `ML_Model_V2/trained_models/OM_BKK_V1`; nothing is
written to the existing binary-model directory.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for horizon, model in intensity_models.items():
    joblib.dump(model, MODEL_DIR / f"intensity_model_{horizon}h.joblib")

(MODEL_DIR / "feature_list.json").write_text(
    json.dumps(FEATURE_COLUMNS, indent=2), encoding="utf-8"
)
(MODEL_DIR / "rainfall_thresholds.json").write_text(
    json.dumps(RAINFALL_THRESHOLDS_MM, indent=2), encoding="utf-8"
)
label_mapping_payload = {
    "label_to_id": LABEL_TO_ID,
    "id_to_label": {str(key): value for key, value in ID_TO_LABEL.items()},
    "ordered_labels": INTENSITY_LABELS,
}
(MODEL_DIR / "intensity_label_mapping.json").write_text(
    json.dumps(label_mapping_payload, indent=2), encoding="utf-8"
)

target_balance.to_csv(MODEL_DIR / "target_balance.csv", index=False)
split_summary.to_csv(MODEL_DIR / "split_summary.csv", index=False)
intensity_results[intensity_results["split"] == "validation"].to_csv(
    MODEL_DIR / "validation_results.csv", index=False
)
intensity_results[intensity_results["split"] == "test"].to_csv(
    MODEL_DIR / "test_results.csv", index=False
)
intensity_class_results[intensity_class_results["split"] == "validation"].to_csv(
    MODEL_DIR / "validation_class_results.csv", index=False
)
intensity_class_results[intensity_class_results["split"] == "test"].to_csv(
    MODEL_DIR / "test_class_results.csv", index=False
)
intensity_confusion_matrices[intensity_confusion_matrices["split"] == "validation"].to_csv(
    MODEL_DIR / "validation_confusion_matrices.csv", index=False
)
intensity_confusion_matrices[intensity_confusion_matrices["split"] == "test"].to_csv(
    MODEL_DIR / "test_confusion_matrices.csv", index=False
)

metadata = {
    "version": "OM_BKK_V1",
    "source_table": SOURCE_TABLE_NAME,
    "precompute_table": PRECOMPUTE_TABLE_NAME,
    "existing_binary_model_dir": str(BINARY_MODEL_DIR),
    "target_definition": "maximum hourly precipitation within t+1 through t+h",
    "conditional_training_filter": "future_max_precipitation_h_mm > 0",
    "horizons": HORIZONS,
    "target_columns": TARGET_COLUMNS,
    "future_max_columns": FUTURE_MAX_COLUMNS,
    "rainfall_thresholds_mm": RAINFALL_THRESHOLDS_MM,
    "label_mapping": label_mapping_payload,
    "feature_columns": FEATURE_COLUMNS,
    "train_boundary": str(train_boundary),
    "validation_boundary": str(validation_boundary),
    "purge_hours_by_horizon": {str(horizon): horizon - 1 for horizon in HORIZONS},
    "sample_rows": SAMPLE_ROWS,
    "training_rows": training_rows,
    "validation_results": intensity_results[intensity_results["split"] == "validation"].to_dict(orient="records"),
    "test_results": intensity_results[intensity_results["split"] == "test"].to_dict(orient="records"),
    "validation_class_results": intensity_class_results[intensity_class_results["split"] == "validation"].to_dict(orient="records"),
    "test_class_results": intensity_class_results[intensity_class_results["split"] == "test"].to_dict(orient="records"),
}
(MODEL_DIR / "OM_BKK_V1_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

print(f"Saved models and evaluation artifacts to {MODEL_DIR}")
display(pd.DataFrame(sorted(path.name for path in MODEL_DIR.iterdir()), columns=["saved_file"]))

## 9. Two-stage inference with the unchanged binary models

The conditional model returns `P(intensity class | rain)`. Overall probabilities are
calculated as `P(rain) × P(intensity class | rain)`. The binary model's configured
probability threshold controls whether the final label is an intensity class or
`No rain`.

In [ ]:
def register_legacy_sklearn_pickle_aliases():
    # The unchanged HistGradientBoosting pickles were created with scikit-learn 1.8,
    # whose compiled loss extension was serialized under the top-level name `_loss`.
    # Registering its current import path lets newer environments read the same files.
    import sys
    try:
        import sklearn._loss._loss as sklearn_loss_extension
    except ImportError:
        return
    sys.modules.setdefault("_loss", sklearn_loss_extension)


def load_existing_binary_rain_models(model_dir=BINARY_MODEL_DIR):
    register_legacy_sklearn_pickle_aliases()
    metadata_path = model_dir / "final_4_metadata.json"
    binary_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    loaded_models = {}
    probability_thresholds = {int(k): float(v) for k, v in binary_metadata["recommended_probability_thresholds"].items()}
    model_plan = {int(k): v for k, v in binary_metadata["model_plan"].items()}

    for horizon in HORIZONS:
        pattern = f"om_bkk_rain_any_next_{horizon}h_neighbor_grid_{model_plan[horizon]}_prob_threshold_*.joblib"
        matches = sorted(model_dir.glob(pattern))
        if len(matches) != 1:
            raise FileNotFoundError(f"Expected one existing binary model for {horizon}h, found: {matches}")
        loaded_models[horizon] = joblib.load(matches[0])
    return loaded_models, probability_thresholds, binary_metadata


binary_rain_models, binary_probability_thresholds, binary_metadata = load_existing_binary_rain_models()


def predict_two_stage(feature_row, horizon):
    if horizon not in HORIZONS:
        raise ValueError(f"horizon must be one of {HORIZONS}")
    x = pd.DataFrame([feature_row]).reindex(columns=FEATURE_COLUMNS).astype("float32")
    if x.isna().any().any():
        missing = x.columns[x.isna().any()].tolist()
        raise ValueError(f"Missing inference features: {missing}")

    rain_probability = float(binary_rain_models[horizon].predict_proba(x)[0, 1])
    rain_threshold = binary_probability_thresholds[horizon]
    rain_predicted = rain_probability >= rain_threshold
    conditional_values = aligned_predict_proba(intensity_models[horizon], x)[0]
    conditional = {
        label: float(conditional_values[class_id])
        for class_id, label in enumerate(INTENSITY_LABELS)
    }
    overall = {
        label: rain_probability * conditional_probability
        for label, conditional_probability in conditional.items()
    }
    final_prediction = (
        max(conditional, key=conditional.get) if rain_predicted else "No rain"
    )
    return {
        "horizon_h": horizon,
        "rain_probability": rain_probability,
        "rain_probability_threshold": rain_threshold,
        "rain_predicted": bool(rain_predicted),
        "conditional_intensity_probabilities_if_rain": conditional,
        "overall_intensity_probabilities": overall,
        "final_prediction": final_prediction,
    }


def print_two_stage_prediction(prediction):
    horizon = prediction["horizon_h"]
    rain_probability = prediction["rain_probability"]
    print(f"Rain probability in next {horizon} hours: {rain_probability:.2f}")
    print("\nConditional intensity probabilities if rain occurs:")
    for label, value in prediction["conditional_intensity_probabilities_if_rain"].items():
        print(f"* {label.removesuffix(' rain')}: {value:.2f}")
    print("\nOverall intensity probabilities:")
    for label, value in prediction["overall_intensity_probabilities"].items():
        conditional_value = prediction["conditional_intensity_probabilities_if_rain"][label]
        print(f"* {label}: {rain_probability:.4f} x {conditional_value:.4f} = {value:.4f}")
    print(f"\nFinal prediction: {prediction['final_prediction']}")

In [ ]:
# Demonstrate the requested six-hour output format on the newest available feature row.
example_feature_row = df.iloc[-1][FEATURE_COLUMNS].to_dict()
six_hour_prediction = predict_two_stage(example_feature_row, horizon=6)
print_two_stage_prediction(six_hour_prediction)
six_hour_prediction

## Result

`OM_BKK_V1` is a conditional second stage: existing models decide rain/no-rain; the
new models describe intensity only when rain is predicted. Overall class probabilities
retain the original rain probability through multiplication, while the final label
remains `No rain` whenever the binary gate is negative.